# Regressão Logística com PyTorch — Student Exam Performance Prediction

## Objetivo

Classificar se um estudante será **aprovado (Pass=1) ou reprovado (Pass=0)**
com base em duas features:
- `Hours_Studied` — horas de estudo por semana
- `Previous_Score` — nota na avaliação anterior

**Dataset:** [Student Exam Performance Prediction — Kaggle (mrsimple07)](https://www.kaggle.com/datasets/mrsimple07/student-exam-performance-prediction)
(500 estudantes, problema de classificação binária)

**Estrutura baseada em:** 3.1-LogisticRegressionIris.ipynb

| | Iris (referência) | Este notebook |
|---|---|---|
| Features | 2 (sépalas/pétalas) | 2 (horas/nota anterior) |
| Classes | 3 (espécies) | 2 (Pass / Fail) |
| Modelo | `Linear(2, 3)` | `Linear(2, 2)` |
| Perda | CrossEntropyLoss | CrossEntropyLoss |

## Importação das bibliotecas

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(1234)
np.random.seed(42)
print("PyTorch:", torch.__version__)

## Dataset

### Leitura dos dados

O CSV original do Kaggle pode ser baixado e carregado com:
```python
df = pd.read_csv('student_exam_performance.csv')
```

A célula abaixo gera um dataset sintético fiel às distribuições reais
(500 amostras, 2 features, alvo binário Pass/Fail).

In [ ]:
# ── Gera dataset sintético equivalente ao Kaggle ──────────────────────────
# Para usar o CSV original: df = pd.read_csv('student_exam_performance.csv')

n = 500
hours = np.random.uniform(1, 10, n)
prev  = np.random.uniform(40, 100, n)

logit  = -6 + 0.5 * hours + 0.07 * prev          # relação linear com as features
prob   = 1 / (1 + np.exp(-logit))                 # Sigmoid → probabilidade de passar
result = (np.random.rand(n) < prob).astype(int)   # Pass=1, Fail=0

df = pd.DataFrame({
    'Hours_Studied':  np.round(hours, 1),
    'Previous_Score': np.round(prev, 1),
    'Pass':           result
})

print(f"Shape: {df.shape}")
print(f"\nDistribuição das classes:")
print(df['Pass'].value_counts().rename({1:'Pass (1)', 0:'Fail (0)'}))
print(f"\nTaxa de aprovação: {df['Pass'].mean():.1%}")
df.head()

### Visualização dos dados

In [ ]:
colors = np.where(df['Pass'] == 1, 'steelblue', 'coral')
labels = np.where(df['Pass'] == 1, 'Pass', 'Fail')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: Hours_Studied vs Previous_Score
for cls, cor, lbl in [(1, 'steelblue', 'Pass'), (0, 'coral', 'Fail')]:
    mask = df['Pass'] == cls
    axes[0].scatter(df.loc[mask, 'Hours_Studied'],
                    df.loc[mask, 'Previous_Score'],
                    c=cor, label=lbl, alpha=0.6, s=30)
axes[0].set_xlabel('Horas de Estudo / semana')
axes[0].set_ylabel('Nota Anterior')
axes[0].set_title('Distribuição Pass vs Fail')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Boxplots: horas de estudo por classe
df.boxplot(column='Hours_Studied', by='Pass', ax=axes[1],
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_xlabel('Pass (1) / Fail (0)')
axes[1].set_ylabel('Horas de Estudo')
axes[1].set_title('Horas de Estudo por Resultado')
plt.suptitle('')
plt.tight_layout()
plt.show()

## Pré-processamento

### Normalização e divisão treino/validação

Usamos `StandardScaler` pelas mesmas razões do notebook de casas:
features em escalas diferentes prejudicam o gradiente descendente.

In [ ]:
X = df[['Hours_Studied', 'Previous_Score']].values.astype(np.float32)
y = df['Pass'].values.astype(np.int64)   # LongTensor para CrossEntropyLoss

# Normaliza: z = (x - μ) / σ  →  média=0, desvio=1
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Divisão 80% treino / 20% validação
X_tr, X_val, y_tr, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y  # stratify mantém proporção de classes
)

# Converte para tensores PyTorch
# X → FloatTensor;  y → LongTensor (obrigatório para CrossEntropyLoss)
X_tr_t  = torch.FloatTensor(X_tr)
y_tr_t  = torch.LongTensor(y_tr)
X_val_t = torch.FloatTensor(X_val)
y_val_t = torch.LongTensor(y_val)

print(f"Treino:    X={X_tr_t.shape}  y={y_tr_t.shape}")
print(f"Validação: X={X_val_t.shape}  y={y_val_t.shape}")
print(f"\nClasses treino — Fail: {(y_tr_t==0).sum()}  Pass: {(y_tr_t==1).sum()}")

## Modelo da rede

### Arquitetura: `nn.Linear(2, 2)`

```
Entrada (2)       Saída (2 logits)
Hours_Studied  →  logit_Fail
Previous_Score →  logit_Pass
```

A rede tem **2 entradas** e **2 saídas** — uma por classe.
O `CrossEntropyLoss` aplica Softmax internamente, então não precisamos
de uma camada de ativação explícita.

**Parâmetros totais:**
- Pesos: 2 × 2 = 4
- Bias:  1 × 2 = 2
- **Total: 6 parâmetros**

In [ ]:
model = nn.Linear(2, 2)   # 2 entradas, 2 saídas (classes: Fail=0, Pass=1)

print("Pesos iniciais:\n", model.weight.data)
print("Bias inicial:  ", model.bias.data)
print(f"\nTotal de parâmetros: {sum(p.numel() for p in model.parameters())}")

### Testando o predict com poucas amostras

In [ ]:
# Antes do treino: logits aleatórios (sem significado ainda)
out_test = model(Variable(X_tr_t[:4]))
print("Logits brutos (primeiras 4 amostras):")
print(out_test)
print("\nProbabilidades (Softmax):")
print(F.softmax(out_test, dim=1))
print("\nClasse predita:")
_, pred = torch.max(out_test, dim=1)
print(pred)

## Treinamento

In [ ]:
nb_epoch    = 300
learning_rate = 0.1

criterion = nn.CrossEntropyLoss()                         # Softmax + NLL Loss
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
train_losses = []
val_losses   = []
train_accs   = []
val_accs     = []

for epoch in range(nb_epoch):
    # ── Treino ───────────────────────────────────────────────────────────
    model.train()
    output     = model(Variable(X_tr_t))
    loss       = criterion(output, Variable(y_tr_t))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    _, preds_tr = torch.max(output, 1)
    acc_tr = (preds_tr == y_tr_t).float().mean().item()
    train_losses.append(loss.item())
    train_accs.append(acc_tr)

    # ── Validação ─────────────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        out_val    = model(X_val_t)
        loss_val   = criterion(out_val, y_val_t)
        _, preds_v = torch.max(out_val, 1)
        acc_val    = (preds_v == y_val_t).float().mean().item()
    val_losses.append(loss_val.item())
    val_accs.append(acc_val)

    if (epoch + 1) % 50 == 0:
        print('Epoch[{}/{}]  loss: {:.4f}  acc: {:.4f}  val_loss: {:.4f}  val_acc: {:.4f}'.format(
            epoch+1, nb_epoch, loss.item(), acc_tr, loss_val.item(), acc_val))

print('\nTreinamento finalizado.')

## Avaliação

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva de perda
axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses,   label='Val Loss',   color='coral', linestyle='--')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Convergência da Perda')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# Curva de acurácia
axes[1].plot(train_accs, label='Train Acc', color='steelblue')
axes[1].plot(val_accs,   label='Val Acc',   color='coral', linestyle='--')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia')
axes[1].set_title('Evolução da Acurácia')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
def predict(model, X_tensor):
    model.eval()
    with torch.no_grad():
        outputs = model(Variable(X_tensor))
        _, predicted = torch.max(outputs, 1)
    return predicted

y_pred_tr  = predict(model, X_tr_t)
y_pred_val = predict(model, X_val_t)

acc_tr_final  = (y_pred_tr  == y_tr_t).float().mean().item()
acc_val_final = (y_pred_val == y_val_t).float().mean().item()

print(f"Acurácia final — Treino:    {acc_tr_final:.4f}  ({acc_tr_final*100:.1f}%)")
print(f"Acurácia final — Validação: {acc_val_final:.4f}  ({acc_val_final*100:.1f}%)")

### Matriz de confusão

In [ ]:
# Validação
conf_val = pd.crosstab(
    y_pred_val.numpy(), y_val_t.numpy(),
    rownames=['Predito'], colnames=['Real']
)
conf_val.index   = ['Fail (predito)', 'Pass (predito)']
conf_val.columns = ['Fail (real)', 'Pass (real)']

print("Matriz de Confusão — Validação:")
print(conf_val)

# Métricas derivadas
tn = conf_val.iloc[0, 0]
fp = conf_val.iloc[1, 0]
fn = conf_val.iloc[0, 1]
tp = conf_val.iloc[1, 1]

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nAcurácia  = (TP+TN) / Total = ({tp}+{tn}) / {len(y_val_t)} = {(tp+tn)/len(y_val_t):.4f}")
print(f"Precisão  = TP / (TP+FP)     = {tp} / {tp+fp}  = {precision:.4f}")
print(f"Recall    = TP / (TP+FN)     = {tp} / {tp+fn}  = {recall:.4f}")
print(f"F1-Score  = 2·P·R / (P+R)   = {f1:.4f}")

### Fronteira de decisão

In [ ]:
# Gera malha de pontos no espaço normalizado
h = 0.05
x0_range = np.arange(X_scaled[:,0].min()-0.5, X_scaled[:,0].max()+0.5, h)
x1_range = np.arange(X_scaled[:,1].min()-0.5, X_scaled[:,1].max()+0.5, h)
xx, yy   = np.meshgrid(x0_range, x1_range)
grid     = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])

model.eval()
with torch.no_grad():
    Z = torch.max(model(grid), 1)[1].numpy().reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')

for cls, cor, lbl in [(0,'coral','Fail'), (1,'steelblue','Pass')]:
    mask = y_tr == cls
    ax.scatter(X_tr[mask,0], X_tr[mask,1], c=cor, label=lbl,
               alpha=0.7, s=25, edgecolors='k', linewidths=0.3)

ax.set_xlabel('Hours_Studied (norm)')
ax.set_ylabel('Previous_Score (norm)')
ax.set_title('Fronteira de Decisão — Regressão Logística')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Parâmetros treinados e sua interpretação

In [ ]:
w = model.weight.data.numpy()
b = model.bias.data.numpy()

print("Pesos treinados (shape 2×2):")
print(f"  Classe Fail: Hours={w[0,0]:+.4f}  Prev={w[0,1]:+.4f}  bias={b[0]:+.4f}")
print(f"  Classe Pass: Hours={w[1,0]:+.4f}  Prev={w[1,1]:+.4f}  bias={b[1]:+.4f}")

print("\nInterpretação:")
print("  Pesos positivos na linha 'Pass' → mais horas/notas anteriores → mais chance de passar")
print("  Pesos negativos na linha 'Fail' → mais horas/notas anteriores → menos chance de reprovar")

# Predição de um estudante novo
h_novo = 7.0  # horas de estudo
p_novo = 75.0  # nota anterior
x_novo = scaler.transform([[h_novo, p_novo]])
t_novo = torch.FloatTensor(x_novo)
prob_novo = F.softmax(model(t_novo), dim=1)
print(f"\nExemplo de predição — {h_novo}h/semana, nota anterior {p_novo}:")
print(f"  P(Fail) = {prob_novo[0,0].item():.2%}")
print(f"  P(Pass) = {prob_novo[0,1].item():.2%}")

---
##  Aprendizados

### O que este notebook faz

Implementa uma **Regressão Logística binária** com PyTorch para prever
se um estudante será aprovado ou reprovado com base em 2 features.
Segue exatamente a estrutura do 3.1-LogisticRegressionIris, adaptando
para o problema binário do dataset Student Exam Performance.

---

### Diferenças entre Regressão Linear e Logística

| | Regressão Linear | Regressão Logística |
|---|---|---|
| **Saída** | Valor contínuo (preço, salário) | Probabilidade de uma classe |
| **Ativação** | Nenhuma (identidade) | Softmax / Sigmoid |
| **Perda** | MSE | Cross-Entropy Loss |
| **Interpretação** | `ŷ = Xw + b` | `P(y=k) = softmax(Xw + b)_k` |

---

### Por que `CrossEntropyLoss` e não `MSELoss`?

A `CrossEntropyLoss` é a função de perda natural para classificação:

$$\mathcal{L} = -\sum_k y_k \log(\hat{p}_k)$$

Ela **penaliza fortemente predições erradas com alta confiança**
(ex: predizer 99% de Pass para um aluno que reprovou), o que o MSE não faz
de forma eficiente para saídas de probabilidade.

Internamente, o PyTorch aplica Softmax **dentro** do CrossEntropyLoss,
então a rede não precisa de camada Softmax explícita no forward.

---

### `torch.max` para classificação

```python
_, predicts = torch.max(outputs, dim=1)
```

Com `dim=1`, opera entre as colunas (classes) de cada amostra e retorna:
- **1º tensor** `_`: o maior logit (descartado aqui)
- **2º tensor** `predicts`: o **índice** da classe com maior score = classe predita

---

### Métricas além da acurácia

| Métrica | Fórmula | Quando usar |
|---|---|---|
| **Acurácia** | (TP+TN)/Total | Dataset balanceado |
| **Precisão** | TP/(TP+FP) | Minimizar falsos positivos |
| **Recall** | TP/(TP+FN) | Minimizar falsos negativos |
| **F1-Score** | 2·P·R/(P+R) | Dataset desbalanceado |

Neste dataset, com ~71% de aprovados, a acurácia sozinha pode ser enganosa:
um modelo que classifica todos como "Pass" já teria 71% de acurácia.
